# 第 11 讲：Scaling 实践与细节

这是 CS336 Lecture 11 的可执行中文学习版。

- 官方来源：`external/cs336-lectures/lecture_11.pdf`
- 官方形式：PDF lecture
- 英文标题：Scaling Case Study And Details

这份 notebook 是学习版，不是逐字翻译。它按官方讲义的结构和节奏整理主线，并在适合的位置加入小实验。


## 0. 本讲你要抓住什么

这一讲继续 scaling：实际调参、初始化、optimizer、学习率、batch、参数化方式，以及如何节省拟合 scaling curve 的 compute。

学习方式：

1. 先读每节的中文解释。
2. 运行对应代码 cell。
3. 改一个参数，观察输出如何变化。
4. 最后用检查点问题自测。


## 1. 本讲路线图

从 scaling law 到实践：hparam tuning、初始化、optimizer、参数化和跨尺度稳定性。


In [1]:
lecture = 11
title = 'Scaling Case Study And Details'
source = 'external/cs336-lectures/lecture_11.pdf'
print(f"Lecture {lecture}: {title}")
print("source:", source)


Lecture 11: Scaling Case Study And Details
source: external/cs336-lectures/lecture_11.pdf


## 2. Best practice 问题

大模型不能盲目调参，只能用小规模实验指导大规模设置。


## 3. 初始化与参数化

不同参数化方式会改变 activation、gradient 和 learning rate 对尺度的敏感度。


## 4. Optimizer 与 learning rate

AdamW、Muon、batch size、warmup、schedule 都可能随规模变化。


## 5. 节省 compute

用小模型/短训练拟合趋势，用代理指标筛选，再把少数设置放大验证。


## 动手实验 1：学习率尺度敏感性的玩具例子

先直接运行，再改一个数字或字符串。


In [2]:
def stable_update(param_scale, lr):
    update = lr * param_scale
    return update < 0.1

for scale in [1, 10, 100]:
    for lr in [1e-3, 1e-2]:
        print("scale", scale, "lr", lr, "stable?", stable_update(scale, lr))


scale 1 lr 0.001 stable? True
scale 1 lr 0.01 stable? True
scale 10 lr 0.001 stable? True
scale 10 lr 0.01 stable? False
scale 100 lr 0.001 stable? False
scale 100 lr 0.01 stable? False


## 动手实验 2：小实验筛选候选超参数

先直接运行，再改一个数字或字符串。


In [3]:
candidates = [
    {"lr": 1e-3, "small_loss": 2.1},
    {"lr": 3e-4, "small_loss": 1.9},
    {"lr": 1e-4, "small_loss": 2.0},
]
best = min(candidates, key=lambda x: x["small_loss"])
print("promote to larger run:", best)


promote to larger run: {'lr': 0.0003, 'small_loss': 1.9}


## 今日检查点

1. 能解释为什么大模型超参数不能只靠试错。
2. 能说出初始化/参数化为什么影响 scaling。
3. 能描述一个小实验到大实验的筛选流程。

如果这些能讲清楚，这一讲的第一轮学习就完成了。
